# 05 · Selección y Entrenamiento de Modelos
**Proyecto:** Sistema de Predicción y Clasificación de la Desnutrición en niños menores de cinco años — Valledupar  
**Dataset:** `113_limpio_unificado.csv` (salida de limpieza y análisis exploratorio)  
**Referencia normativa:** Resolución 2465 de 2016 (indicadores P/T y T/E), Resolución 2350 de 2020

---

## Objetivos de este notebook
1. Preparar las features y las variables objetivo para ML  
2. Dividir el dataset en conjuntos de entrenamiento y prueba (estratificado)  
3. Entrenar y comparar 4 algoritmos de clasificación  
4. Evaluar rendimiento con métricas estándar (accuracy, F1, AUC-ROC)  
5. Seleccionar el mejor modelo e interpretar sus resultados  

## Variables objetivo
Se trabajan **dos tareas de clasificación binaria** alineadas con los indicadores de la Resolución 2465/2016:

| Variable objetivo | Descripción | Codificación |
|---|---|---|
| `target_peso` | Desnutrición según **Peso/Talla** | 0 = Normal (clas_peso=2), 1 = Cualquier malnutrición |
| `target_talla` | Desnutrición según **Talla/Edad** | 0 = Normal (clas_talla=1), 1 = Retraso en talla |

> ⚠️ **Nota:** Los z-scores (`zscore_pt`, `zscore_te`) **no se incluyen como features** porque son la base directa de la clasificación — usarlos generaría data leakage. El modelo aprende desde variables clínicas y socioeconómicas observables.

## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

# Métricas
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, f1_score, accuracy_score, balanced_accuracy_score,
    precision_score, recall_score
)

# Reproducibilidad
SEED = 42
np.random.seed(SEED)

# Estilo de gráficas
sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.1)
COLORES = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12', '#9b59b6']

print('Librerías cargadas correctamente ✓')

## 2. Carga y revisión del dataset

In [ ]:
df_raw = pd.read_csv('113_limpio_unificado.csv')
print(f'Dimensiones del dataset: {df_raw.shape[0]} registros × {df_raw.shape[1]} columnas')
df_raw.head()

In [ ]:
# Eliminar filas sin variable objetivo
df = df_raw.dropna(subset=['clas_peso', 'clas_talla']).copy()
print(f'Registros tras eliminar nulos en targets: {len(df)} (eliminados: {len(df_raw) - len(df)})')
print()
print('Distribución clas_peso (Peso/Talla):')
print(df['clas_peso'].value_counts().sort_index().rename({1.0:'Bajo peso', 2.0:'Normal', 3.0:'Sobrepeso', 4.0:'Obesidad', 5.0:'Desnutrición aguda moderada', 6.0:'Desnutrición aguda severa'}))
print()
print('Distribución clas_talla (Talla/Edad):')
print(df['clas_talla'].value_counts().sort_index().rename({1.0:'Talla adecuada', 2.0:'Riesgo de baja talla', 3.0:'Baja talla'}))

## 3. Ingeniería de features y construcción de variables objetivo

In [ ]:
# ── Features seleccionadas ──────────────────────────────────────────────────
# Se excluyen:
#   - zscore_pt, zscore_te       → generan data leakage (determinan el target directamente)
#   - clas_peso, clas_talla      → son los targets
#   - columnas de código/ID      → sin poder predictivo (cod_pais_r, cod_dpto_o, etc.)
#   - nom_grupo_, municipio_orig → alta cardinalidad, representada por per_etn_ y area_
#   - talla_nac                  → 61.7% nulos, demasiado incompleta

FEATURES = [
    # Demográficas
    'edad_',          # Edad en meses
    'uni_med_',       # Unidad de medida de la edad
    'sexo_',          # Sexo (M/F)
    'per_etn_',       # Pertenencia étnica (codificada)
    'estrato_',       # Estrato socioeconómico
    'area_',          # Área (rural/urbana)
    # Antecedentes al nacimiento
    'peso_nac',       # Peso al nacer
    'edad_ges',       # Edad gestacional
    # Mediciones actuales
    'peso_act',       # Peso actual
    'talla_act',      # Talla actual
    'imc',            # Índice de masa corporal
    # Hallazgos clínicos
    'edema',
    'delgadez',
    'palidez',
    'piel_rese',
    'hiperpigm',
    'cambios_cabello',
]

print(f'Total features seleccionadas: {len(FEATURES)}')
print(FEATURES)

In [ ]:
# ── Codificación de variables categóricas ───────────────────────────────────
df['sexo_'] = (df['sexo_'] == 'M').astype(int)  # M=1, F=0

# ── Imputación de valores faltantes (mediana) ───────────────────────────────
X_all = df[FEATURES].copy()
for col in X_all.columns:
    if X_all[col].isnull().any():
        mediana = X_all[col].median()
        X_all[col] = X_all[col].fillna(mediana)
        print(f'  Imputados {df[col].isnull().sum()} nulos en "{col}" con mediana={mediana:.2f}')

print('\nNulos restantes:', X_all.isnull().sum().sum())

In [ ]:
# ── Construcción de variables objetivo binarias ──────────────────────────────
# TARGET 1: Desnutrición por Peso/Talla
#   0 = Normal (clas_peso == 2)
#   1 = Cualquier malnutrición (bajo peso, sobrepeso, obesidad, desnutrición aguda)
y_peso  = (df['clas_peso'] != 2).astype(int)

# TARGET 2: Desnutrición por Talla/Edad (retraso en talla / stunting)
#   0 = Talla adecuada (clas_talla == 1)
#   1 = Riesgo de baja talla o baja talla (clas_talla == 2 o 3)
y_talla = (df['clas_talla'] != 1).astype(int)

print('TARGET 1 – Desnutrición Peso/Talla:')
counts_p = y_peso.value_counts()
print(f'  Normal (0): {counts_p.get(0,0)} ({counts_p.get(0,0)/len(y_peso)*100:.1f}%)')
print(f'  Malnutrición (1): {counts_p.get(1,0)} ({counts_p.get(1,0)/len(y_peso)*100:.1f}%)')

print('\nTARGET 2 – Retraso en Talla/Edad:')
counts_t = y_talla.value_counts()
print(f'  Talla adecuada (0): {counts_t.get(0,0)} ({counts_t.get(0,0)/len(y_talla)*100:.1f}%)')
print(f'  Retraso en talla (1): {counts_t.get(1,0)} ({counts_t.get(1,0)/len(y_talla)*100:.1f}%)')

## 4. División del dataset (Train / Test)

In [ ]:
# División 80% entrenamiento / 20% prueba, estratificada por target
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_all, y_peso, test_size=0.2, random_state=SEED, stratify=y_peso
)

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_all, y_talla, test_size=0.2, random_state=SEED, stratify=y_talla
)

print('=== División Train/Test ===')
print(f'  Entrenamiento: {X_train_p.shape[0]} registros ({X_train_p.shape[0]/len(X_all)*100:.0f}%)')
print(f'  Prueba:        {X_test_p.shape[0]} registros ({X_test_p.shape[0]/len(X_all)*100:.0f}%)')

# Verificar estratificación
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, (y_tr, y_te, titulo) in zip(axes, [
    (y_train_p, y_test_p, 'Peso/Talla'),
    (y_train_t, y_test_t, 'Talla/Edad')
]):
    pct_train = y_tr.value_counts(normalize=True).sort_index()
    pct_test  = y_te.value_counts(normalize=True).sort_index()
    x = np.arange(2)
    w = 0.35
    ax.bar(x - w/2, pct_train.values * 100, w, label='Train', color=COLORES[0])
    ax.bar(x + w/2, pct_test.values * 100,  w, label='Test',  color=COLORES[1])
    ax.set_xticks(x)
    ax.set_xticklabels(['Normal (0)', 'Malnutrición (1)'])
    ax.set_ylabel('Porcentaje (%)')
    ax.set_title(f'Estratificación – {titulo}')
    ax.legend()
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

plt.tight_layout()
plt.savefig('split_estratificacion.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ La división estratificada mantiene proporciones similares en train y test')

## 5. Definición de modelos

Se evalúan cuatro algoritmos ampliamente utilizados en la literatura de predicción nutricional:

| Modelo | Ventaja principal | Referencia en la literatura |
|---|---|---|
| Regresión Logística | Interpretable, baseline sólido | Polanía Macías, 2023 |
| Árbol de Decisión | Altamente interpretable, reglas claras | CRISP-DM estándar |
| Random Forest | Robusto, maneja bien el desbalance | Hincapié & Gutiérrez (Antioquia) |
| Gradient Boosting | Alto rendimiento predictivo | Centro de Salud Uliachin, Perú |

> Se usa `class_weight='balanced'` para contrarrestar el desbalance de clases sin resampling artificial.

In [ ]:
MODELOS = {
    'Regresión Logística': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            random_state=SEED,
            solver='lbfgs'
        ))
    ]),
    'Árbol de Decisión': DecisionTreeClassifier(
        max_depth=10,
        min_samples_leaf=10,
        class_weight='balanced',
        random_state=SEED
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=5,
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=5,
        random_state=SEED
    ),
}

print('Modelos definidos:')
for nombre in MODELOS:
    print(f'  ✓ {nombre}')

## 6. Entrenamiento y evaluación – TARGET 1: Desnutrición Peso/Talla

In [ ]:
def evaluar_modelo(nombre, modelo, X_tr, X_te, y_tr, y_te, cv_folds=5):
    """Entrena, valida con CV y evalúa en test. Retorna dict con métricas."""
    # Entrenamiento
    modelo.fit(X_tr, y_tr)
    y_pred = modelo.predict(X_te)
    y_prob = modelo.predict_proba(X_te)[:, 1] if hasattr(modelo, 'predict_proba') else None

    # Métricas en test
    acc   = accuracy_score(y_te, y_pred)
    bal   = balanced_accuracy_score(y_te, y_pred)
    f1_w  = f1_score(y_te, y_pred, average='weighted')
    prec  = precision_score(y_te, y_pred, average='weighted', zero_division=0)
    rec   = recall_score(y_te, y_pred, average='weighted')
    auc   = roc_auc_score(y_te, y_prob) if y_prob is not None else np.nan

    # Validación cruzada (F1 ponderado)
    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=SEED)
    cv_scores = cross_val_score(modelo, X_tr, y_tr, cv=cv, scoring='f1_weighted', n_jobs=-1)

    return {
        'Modelo': nombre,
        'Accuracy': acc,
        'Balanced Acc': bal,
        'Precision': prec,
        'Recall': rec,
        'F1 (weighted)': f1_w,
        'AUC-ROC': auc,
        'CV F1 Media': cv_scores.mean(),
        'CV F1 Std': cv_scores.std(),
        'modelo_obj': modelo,
        'y_pred': y_pred,
        'y_prob': y_prob,
    }

print('Función de evaluación definida ✓')
print()
print('=== Entrenando modelos para TARGET 1 – Desnutrición Peso/Talla ===')
print()

resultados_peso = {}
for nombre, modelo in MODELOS.items():
    print(f'  Entrenando: {nombre}...')
    res = evaluar_modelo(nombre, modelo, X_train_p, X_test_p, y_train_p, y_test_p)
    resultados_peso[nombre] = res
    print(f'    Accuracy={res["Accuracy"]:.3f} | F1={res["F1 (weighted)"]:.3f} | AUC={res["AUC-ROC"]:.3f} | CV_F1={res["CV F1 Media"]:.3f} ± {res["CV F1 Std"]:.3f}')

print('\n✓ Entrenamiento completado')

In [ ]:
# ── Tabla comparativa TARGET 1 ───────────────────────────────────────────────
cols_mostrar = ['Modelo','Accuracy','Balanced Acc','Precision','Recall','F1 (weighted)','AUC-ROC','CV F1 Media','CV F1 Std']
df_res_peso = pd.DataFrame([{k: v for k, v in r.items() if k in cols_mostrar} for r in resultados_peso.values()])
df_res_peso = df_res_peso.set_index('Modelo').sort_values('F1 (weighted)', ascending=False)
df_res_peso_display = df_res_peso.copy()
for col in df_res_peso_display.columns:
    df_res_peso_display[col] = df_res_peso_display[col].map('{:.4f}'.format)
print('=== Comparativa de modelos – Peso/Talla ===')
df_res_peso_display

## 7. Entrenamiento y evaluación – TARGET 2: Retraso en Talla/Edad

In [ ]:
# Re-instanciar los modelos para evitar contaminación de estado
MODELOS_T = {
    'Regresión Logística': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED, solver='lbfgs'))
    ]),
    'Árbol de Decisión': DecisionTreeClassifier(max_depth=10, min_samples_leaf=10, class_weight='balanced', random_state=SEED),
    'Random Forest': RandomForestClassifier(n_estimators=200, min_samples_leaf=5, class_weight='balanced', random_state=SEED, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=SEED),
}

print('=== Entrenando modelos para TARGET 2 – Retraso en Talla/Edad ===')
print()

resultados_talla = {}
for nombre, modelo in MODELOS_T.items():
    print(f'  Entrenando: {nombre}...')
    res = evaluar_modelo(nombre, modelo, X_train_t, X_test_t, y_train_t, y_test_t)
    resultados_talla[nombre] = res
    print(f'    Accuracy={res["Accuracy"]:.3f} | F1={res["F1 (weighted)"]:.3f} | AUC={res["AUC-ROC"]:.3f} | CV_F1={res["CV F1 Media"]:.3f} ± {res["CV F1 Std"]:.3f}')

df_res_talla = pd.DataFrame([{k: v for k, v in r.items() if k in cols_mostrar} for r in resultados_talla.values()])
df_res_talla = df_res_talla.set_index('Modelo').sort_values('F1 (weighted)', ascending=False)
df_res_talla_display = df_res_talla.copy()
for col in df_res_talla_display.columns:
    df_res_talla_display[col] = df_res_talla_display[col].map('{:.4f}'.format)
print()
print('=== Comparativa de modelos – Talla/Edad ===')
df_res_talla_display

## 8. Visualización comparativa de resultados

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metricas_plot = ['Accuracy', 'Balanced Acc', 'F1 (weighted)', 'AUC-ROC']

for ax, (df_res, titulo) in zip(axes, [
    (df_res_peso, 'TARGET 1 – Desnutrición Peso/Talla'),
    (df_res_talla, 'TARGET 2 – Retraso en Talla/Edad')
]):
    x = np.arange(len(df_res.index))
    width = 0.18
    for i, metrica in enumerate(metricas_plot):
        vals = df_res[metrica].astype(float)
        bars = ax.bar(x + i * width, vals, width, label=metrica, color=COLORES[i])
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                    f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7.5, rotation=90)
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(df_res.index, rotation=15, ha='right', fontsize=9)
    ax.set_ylim(0, 1.12)
    ax.set_ylabel('Valor de la métrica')
    ax.set_title(titulo, fontsize=10, fontweight='bold')
    ax.legend(loc='lower right', fontsize=8)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))

plt.suptitle('Comparativa de rendimiento de modelos de clasificación', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('comparativa_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Curvas ROC – TARGET 1 ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (resultados, y_te, titulo) in zip(axes, [
    (resultados_peso,  y_test_p, 'Desnutrición Peso/Talla'),
    (resultados_talla, y_test_t, 'Retraso en Talla/Edad')
]):
    for (nombre, res), color in zip(resultados.items(), COLORES):
        if res['y_prob'] is not None:
            fpr, tpr, _ = roc_curve(y_te, res['y_prob'])
            ax.plot(fpr, tpr, color=color, lw=2,
                    label=f"{nombre} (AUC={res['AUC-ROC']:.3f})")
    ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.6, label='Clasificador aleatorio')
    ax.set_xlabel('Tasa de Falsos Positivos')
    ax.set_ylabel('Tasa de Verdaderos Positivos (Sensibilidad)')
    ax.set_title(f'Curvas ROC – {titulo}', fontweight='bold')
    ax.legend(loc='lower right', fontsize=8.5)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('curvas_roc.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Selección del mejor modelo

El criterio de selección combina:
- **AUC-ROC** (discriminación global)  
- **F1 ponderado** (balance entre precisión y recall con clases desbalanceadas)  
- **Balanced Accuracy** (sensible al desbalance)  
- **CV F1** (generalización estimada)

In [ ]:
def seleccionar_mejor(resultados, etiqueta):
    # Score compuesto: promedio de AUC, F1 y Balanced Accuracy
    scores = {}
    for nombre, r in resultados.items():
        auc = r['AUC-ROC'] if not np.isnan(r['AUC-ROC']) else 0
        scores[nombre] = (auc + r['F1 (weighted)'] + r['Balanced Acc'] + r['CV F1 Media']) / 4
    mejor = max(scores, key=scores.get)
    print(f'Ranking de modelos – {etiqueta}:')
    for i, (nombre, score) in enumerate(sorted(scores.items(), key=lambda x: -x[1]), 1):
        marca = ' ← MEJOR' if nombre == mejor else ''
        print(f'  {i}. {nombre}: score={score:.4f}{marca}')
    return mejor

mejor_peso  = seleccionar_mejor(resultados_peso,  'Peso/Talla')
print()
mejor_talla = seleccionar_mejor(resultados_talla, 'Talla/Edad')

print(f'\n✅ Modelo seleccionado para Peso/Talla:  {mejor_peso}')
print(f'✅ Modelo seleccionado para Talla/Edad:  {mejor_talla}')

## 10. Matrices de confusión de los mejores modelos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, (nombre, resultados, y_te, etiqueta) in zip(axes, [
    (mejor_peso,  resultados_peso,  y_test_p, 'Peso/Talla'),
    (mejor_talla, resultados_talla, y_test_t, 'Talla/Edad')
]):
    y_pred = resultados[nombre]['y_pred']
    cm = confusion_matrix(y_te, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Normal (0)', 'Malnutrición (1)']
    )
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{nombre}\nTarget: {etiqueta}', fontsize=10, fontweight='bold')

plt.suptitle('Matrices de Confusión – Mejores Modelos', fontsize=12, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('matrices_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

print()
print(f'=== Reporte completo – {mejor_peso} (Peso/Talla) ===')
print(classification_report(y_test_p, resultados_peso[mejor_peso]['y_pred'],
                             target_names=['Normal', 'Malnutrición']))
print(f'=== Reporte completo – {mejor_talla} (Talla/Edad) ===')
print(classification_report(y_test_t, resultados_talla[mejor_talla]['y_pred'],
                             target_names=['Talla adecuada', 'Retraso en talla']))

## 11. Importancia de variables (modelos basados en árboles)

In [ ]:
NOMBRES_FEATURES = {
    'edad_':           'Edad (meses)',
    'uni_med_':        'Unidad medida edad',
    'sexo_':           'Sexo',
    'per_etn_':        'Pertenencia étnica',
    'estrato_':        'Estrato socioeconómico',
    'area_':           'Área (rural/urbana)',
    'peso_nac':        'Peso al nacer',
    'edad_ges':        'Edad gestacional',
    'peso_act':        'Peso actual',
    'talla_act':       'Talla actual',
    'imc':             'IMC',
    'edema':           'Edema',
    'delgadez':        'Delgadez visible',
    'palidez':         'Palidez',
    'piel_rese':       'Piel reseca',
    'hiperpigm':       'Hiperpigmentación',
    'cambios_cabello': 'Cambios en cabello',
}

def plot_importancia(modelo_obj, titulo, ax):
    # Para Pipeline, extraer el clasificador
    clf = modelo_obj.named_steps['clf'] if hasattr(modelo_obj, 'named_steps') else modelo_obj
    if not hasattr(clf, 'feature_importances_'):
        ax.text(0.5, 0.5, 'Este modelo no soporta\nimportancia de variables', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(titulo)
        return
    importancias = clf.feature_importances_
    nombres = [NOMBRES_FEATURES.get(f, f) for f in FEATURES]
    df_imp = pd.Series(importancias, index=nombres).sort_values(ascending=True)
    df_imp.tail(12).plot(kind='barh', ax=ax, color='#3498db', edgecolor='white')
    ax.set_xlabel('Importancia relativa')
    ax.set_title(titulo, fontweight='bold')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

plot_importancia(
    resultados_peso[mejor_peso]['modelo_obj'],
    f'Top features – {mejor_peso}\n(Desnutrición Peso/Talla)',
    axes[0]
)
plot_importancia(
    resultados_talla[mejor_talla]['modelo_obj'],
    f'Top features – {mejor_talla}\n(Retraso en Talla/Edad)',
    axes[1]
)

plt.tight_layout()
plt.savefig('importancia_variables.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Validación cruzada detallada (5-Fold Stratified)

In [ ]:
def cv_detallado(modelo_obj, X, y, nombre, cv_folds=5):
    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=SEED)
    metricas = ['accuracy', 'f1_weighted', 'roc_auc', 'balanced_accuracy']
    print(f'\n--- Validación cruzada {cv_folds}-Fold: {nombre} ---')
    resultados_cv = {}
    for m in metricas:
        scores = cross_val_score(modelo_obj, X, y, cv=cv, scoring=m, n_jobs=-1)
        resultados_cv[m] = scores
        print(f'  {m:20s}: {scores.mean():.4f} ± {scores.std():.4f}  |  folds: {[f"{s:.3f}" for s in scores]}')
    return resultados_cv

print('==== TARGET 1 – Desnutrición Peso/Talla ====')
cv_peso = cv_detallado(resultados_peso[mejor_peso]['modelo_obj'], X_all, y_peso, mejor_peso)

print('\n==== TARGET 2 – Retraso en Talla/Edad ====')
cv_talla = cv_detallado(resultados_talla[mejor_talla]['modelo_obj'], X_all, y_talla, mejor_talla)

## 13. Guardar modelos entrenados

In [ ]:
import pickle
import os

os.makedirs('modelos', exist_ok=True)

# Guardar modelos
with open('modelos/modelo_peso_talla.pkl', 'wb') as f:
    pickle.dump(resultados_peso[mejor_peso]['modelo_obj'], f)

with open('modelos/modelo_talla_edad.pkl', 'wb') as f:
    pickle.dump(resultados_talla[mejor_talla]['modelo_obj'], f)

# Guardar nombres de features
with open('modelos/features.pkl', 'wb') as f:
    pickle.dump(FEATURES, f)

# Guardar tabla de resultados
df_res_peso.to_csv('modelos/resultados_peso_talla.csv')
df_res_talla.to_csv('modelos/resultados_talla_edad.csv')

print('Modelos guardados en carpeta /modelos:')
for f in os.listdir('modelos'):
    print(f'  ✓ {f}')

## 14. Resumen ejecutivo

In [ ]:
print('=' * 65)
print('RESUMEN EJECUTIVO – Selección y Entrenamiento de Modelos')
print('Sistema de Predicción de Desnutrición Infantil · Valledupar')
print('=' * 65)

for etiq, nombre, resultados, y_te in [
    ('Peso/Talla',  mejor_peso,  resultados_peso,  y_test_p),
    ('Talla/Edad',  mejor_talla, resultados_talla, y_test_t)
]:
    r = resultados[nombre]
    print(f'\n◆ TARGET: {etiq}')
    print(f'  Mejor modelo : {nombre}')
    print(f'  Accuracy     : {r["Accuracy"]:.4f}')
    print(f'  Balanced Acc : {r["Balanced Acc"]:.4f}')
    print(f'  F1 (weighted): {r["F1 (weighted)"]:.4f}')
    print(f'  AUC-ROC      : {r["AUC-ROC"]:.4f}')
    print(f'  CV F1 (5-fold): {r["CV F1 Media"]:.4f} ± {r["CV F1 Std"]:.4f}')

print()
print('Archivos generados:')
print('  📊 comparativa_modelos.png   – Barras de métricas por modelo')
print('  📈 curvas_roc.png            – Curvas ROC de todos los modelos')
print('  🔲 matrices_confusion.png   – Matrices de confusión')
print('  📌 importancia_variables.png – Top features')
print('  💾 modelos/                  – Modelos .pkl listos para inferencia')
print()
print('Siguiente paso: 06_dashboard_resultados.ipynb')
print('=' * 65)